## Load data

In [1]:
import numpy as np
import pandas as pd


In [2]:
def read_texts_from_dir(dir_path):
    """
    Reads the texts from a given directory and saves them in the pd.DataFrame with columns ['id', 'file_1', 'file_2'].

    Params:
      dir_path (str): path to the directory with data
    """
    # Count number of directories in the provided path
    dir_count = sum(
        os.path.isdir(os.path.join(root, d))
        for root, dirs, _ in os.walk(dir_path)
        for d in dirs
    )
    data = [0 for _ in range(dir_count)]
    print(f"Number of directories: {dir_count}")

    # For each directory, read both file_1.txt and file_2.txt and save results to the list
    i = 0
    for folder_name in sorted(os.listdir(dir_path)):
        folder_path = os.path.join(dir_path, folder_name)
        if os.path.isdir(folder_path):
            try:
                with open(
                    os.path.join(folder_path, "file_1.txt"), "r", encoding="utf-8"
                ) as f1:
                    text1 = f1.read().strip()
                with open(
                    os.path.join(folder_path, "file_2.txt"), "r", encoding="utf-8"
                ) as f2:
                    text2 = f2.read().strip()
                index = int(folder_name[-4:])
                data[i] = (index, text1, text2)
                i += 1
            except Exception as e:
                print(f"Error reading directory {folder_name}: {e}")

    # Change list with results into pandas DataFrame
    df = pd.DataFrame(data, columns=["id", "file_1", "file_2"]).set_index("id")
    return df

In [4]:
import os

df_train = read_texts_from_dir(r"C:\workspace\AI\Machine learning\CTAI_MachineLearning\data\train")
df_test = read_texts_from_dir(r"C:\workspace\AI\Machine learning\CTAI_MachineLearning\data\test")
train_labels = pd.read_csv(r'C:\workspace\AI\Machine learning\CTAI_MachineLearning\data\train.csv')


Number of directories: 95
Number of directories: 1068


In [5]:
df_train['label'] = train_labels['real_text_id'].values - 1
df_train.head()

,file_1,file_2,label
id,,,
0,The VIRSA (Visible Infrared Survey Telescope A...,The China relay network has released a signifi...,0
1,China\nThe goal of this project involves achie...,The project aims to achieve an accuracy level ...,1
2,Scientists can learn about how galaxies form a...,Dinosaur eggshells offer clues about what dino...,0
3,China\nThe study suggests that multiple star s...,The importance for understanding how stars evo...,1
4,Dinosaur Rex was excited about his new toy set...,Analyzing how fast stars rotate within a galax...,1


In [6]:
df_train.loc[10] ## file 2 kô có nội dung

file_1    To determine how old stars are within R136's c...
file_2                                                     
label                                                     0
Name: 10, dtype: object

In [7]:
df_train.drop(10, axis=0, inplace=True)
df_train.reset_index(drop=True, inplace=True)
df_train.shape

(94, 3)

## Clean Text

In [8]:
import re

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # 2. Xóa các ký tự không mong muốn nhưng giữ lại ' và - nếu ở trong từ
    #   - Cho phép: chữ, số, khoảng trắng, ', -
    text = re.sub(r"[^A-Za-z0-9\s'\-]", " ", text)
    # 3. Chuẩn hoá khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_train['file_1'] = df_train['file_1'].apply(clean_text)
df_train['file_2'] = df_train['file_2'].apply(clean_text)
df_test['file_1'] = df_test['file_1'].apply(clean_text)
df_test['file_2'] = df_test['file_2'].apply(clean_text)



In [9]:
df_train.head()

,file_1,file_2,label
0,The VIRSA Visible Infrared Survey Telescope Ar...,The China relay network has released a signifi...,0
1,China The goal of this project involves achiev...,The project aims to achieve an accuracy level ...,1
2,Scientists can learn about how galaxies form a...,Dinosaur eggshells offer clues about what dino...,0
3,China The study suggests that multiple star sy...,The importance for understanding how stars evo...,1
4,Dinosaur Rex was excited about his new toy set...,Analyzing how fast stars rotate within a galax...,1


## Prepare Data for train

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim


In [11]:
import spacy
nlp = spacy.load("en_core_web_sm")

def tokenize_fn(text1, text2, max_length=512):
    tokens1 = [token.text for token in nlp.tokenizer(text1)]
    tokens2 = [token.text for token in nlp.tokenizer(text2)]
    combined_tokens = None
    if len(tokens1) + len(tokens2) + 1 > max_length:
        if len(tokens1) < max_length // 2:
           combined_tokens = ['<CLS>'] + tokens1 + ['<SEP>'] + tokens2[:max_length - len(tokens1) - 2]
        elif len(tokens2) < max_length // 2:
           combined_tokens = ['<CLS>'] + tokens1[:max_length - len(tokens2) - 2] + ['<SEP>'] + tokens2
        else:
           combined_tokens = ['<CLS>'] + tokens1[:max_length // 2 - 1] + ['<SEP>'] + tokens2[:max_length // 2 - 1]
    else:
        combined_tokens = ['<CLS>'] + tokens1 + ['<SEP>'] + tokens2 + ['<PAD>'] * (max_length - len(tokens1) - len(tokens2) - 2)

    return combined_tokens


class Vocab:
    def __init__(self):
        self.word2idx = {'<CLS>': 0, '<SEP>': 1, '<PAD>': 2, '<UNK>': 3}
        self.idx2word = {0: "<CLS>", 1: "<SEP>", 2: "<PAD>", 3: "<UNK>"}
        

    def build_vocab(self, df_data):
        idx = 4
        for _, row in df_data.iterrows():
            tokens = tokenize_fn(row['file_1'], row['file_2'])
            for token in tokens:
                if token not in self.word2idx:
                    self.word2idx[token] = idx
                    self.idx2word[idx] = token
                    idx += 1
    def encode(self, tokens):
        return [self.word2idx.get(token, self.word2idx['<UNK>']) for token in tokens]

In [12]:
class TextDataset(Dataset):
    def __init__(self, dataframe, vocab, tokenizer_fn, max_length=512):
        self.dataframe = dataframe
        self.tokenizer_fn = tokenizer_fn
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        text1 = row['file_1']
        text2 = row['file_2']
        # Default to None if label not present
        label = row.get('label', None)

        # Tokenize and encode the texts
        tokens = self.tokenizer_fn(
            text1,
            text2,
            max_length=self.max_length
        )
        encode = self.vocab.encode(tokens)

        if label is not None:
            return torch.tensor(encode, dtype=torch.long), torch.tensor(label, dtype=torch.long)
        return torch.tensor(encode, dtype=torch.long)

In [13]:
from sklearn.model_selection import train_test_split

max_length = 1024 
vocab = Vocab()


train_data, val_data = train_test_split(df_train, test_size=0.2, random_state=42)
vocab.build_vocab(train_data)

train_dataset = TextDataset(train_data, vocab, tokenize_fn, max_length=max_length)
val_dataset = TextDataset(val_data, vocab, tokenize_fn, max_length=max_length)
test_dataset = TextDataset(df_test, vocab, tokenize_fn, max_length=max_length)


In [14]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

len(train_loader), len(val_loader), len(test_loader)

(10, 3, 134)

## Modeling

In [15]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128, output_dim=2, n_layers=2, bidirectional=True, dropout=0.3):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=2)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=n_layers, bidirectional=bidirectional, batch_first=True, dropout=dropout)
        self.fc1 = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, 64)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, output_dim)


    def forward(self, x):
        x = self.dropout(self.embedding(x))
        h0 = torch.zeros(self.lstm.num_layers * (2 if self.lstm.bidirectional else 1), x.size(0), self.lstm.hidden_size).to(x.device)
        c0 = torch.zeros(self.lstm.num_layers * (2 if self.lstm.bidirectional else 1), x.size(0), self.lstm.hidden_size).to(x.device)
        h, _ = self.lstm(x, (h0, c0))
        x = h[:, -1, :]
        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)
        return x


In [16]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LSTMClassifier(embedding_dim=512, hidden_dim=256, n_layers=2, bidirectional=False, dropout=0.3, vocab_size=len(train_dataset.vocab.word2idx)).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [17]:

def train_step(model, data_loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0
    for inputs, labels in data_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(data_loader)
def eval_step(model, data_loader, loss_fn, device):
    model.eval()
    total_loss = 0
    correct = 0
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
    accuracy = correct / len(data_loader.dataset)
    return total_loss / len(data_loader), accuracy

def predict(model, data_loader, device):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for inputs in data_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
    return all_preds

def train_and_validate(model, train_loader, val_loader, loss_fn, optimizer, device, epochs=5):
    for epoch in range(epochs):
        train_loss = train_step(model, train_loader, loss_fn, optimizer, device)
        val_loss, val_accuracy = eval_step(model, val_loader, loss_fn, device)
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")

In [18]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 4,083,394
Trainable parameters: 4,083,394


In [19]:
train_and_validate(model, train_loader, val_loader, loss_fn, optimizer, device, epochs=10)

Epoch 1/10, Train Loss: 0.6839, Val Loss: 0.7091, Val Accuracy: 0.4737
Epoch 2/10, Train Loss: 0.6609, Val Loss: 0.8305, Val Accuracy: 0.4737
Epoch 3/10, Train Loss: 0.6370, Val Loss: 0.9580, Val Accuracy: 0.5263
Epoch 4/10, Train Loss: 0.5912, Val Loss: 0.8069, Val Accuracy: 0.5789
Epoch 5/10, Train Loss: 0.6267, Val Loss: 0.7066, Val Accuracy: 0.5789
Epoch 6/10, Train Loss: 0.5595, Val Loss: 0.6987, Val Accuracy: 0.5789
Epoch 7/10, Train Loss: 0.5704, Val Loss: 0.6190, Val Accuracy: 0.6316
Epoch 8/10, Train Loss: 0.5500, Val Loss: 0.6143, Val Accuracy: 0.6316
Epoch 9/10, Train Loss: 0.5501, Val Loss: 0.8284, Val Accuracy: 0.5789
Epoch 10/10, Train Loss: 0.5503, Val Loss: 0.8609, Val Accuracy: 0.5789


In [20]:
# preds = predict(model, test_loader, device)
# submission = pd.DataFrame({'id': df_test.index, 'real_text_id': np.array(preds) + 1})
# submission.to_csv('submission.csv', index=False)